# Try Embodimetry in 2 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thrmnn/embodimetry/blob/main/notebooks/try_embodimetry.ipynb)

**[Embodimetry](https://github.com/thrmnn/embodimetry)** is a public, reproducible instrument for embodied-AI policies. Every leaderboard number is a binary-outcome estimate with a confidence interval, anchored to a pinned `lerobot` release and reproducible from a seed.

This notebook runs a **real eval cell live on a free Colab CPU** — no GPU, no Hugging Face login, no checkpoint download. We evaluate the two *weights-free floors* of the benchmark on the PushT task:

- **`no_op`** — always emits a zero action (the do-nothing floor)
- **`random`** — uniform random actions in `[-1, 1]` (the chance floor)

Both run on CPU in well under a minute and produce a **real success rate with a Wilson 95% confidence interval** — the exact same `embodimetry.eval.run_cell_from_specs` + `embodimetry.stats.wilson_ci` API the full sweep and the leaderboard use.

> Why floors first? A benchmark is only trustworthy if its *zero-skill baselines* score near zero. Reproducing the floors live, in your own runtime, is the cheapest honest proof that the contract is sound — before you ever download a 1 GB checkpoint.

## 1. Install (~2–4 min on Colab)

We `git clone` the repo (for the `configs/*.yaml` registries) and `pip install` it with the `[sim]` extra, which pulls in `gym-pusht` + `mujoco`. The heavy part is `lerobot==0.5.1` + `torch` — grab a coffee. The eval itself is CPU-only and fast.

In [ ]:
# Clone (for configs/) and install the package with sim envs. Runs on a free CPU runtime.
!git clone --depth 1 https://github.com/thrmnn/embodimetry.git
%pip install -q -e "./embodimetry[sim]"

# MuJoCo needs a headless GL backend on Colab.
import os

os.environ.setdefault("MUJOCO_GL", "egl")
print("install done")

## 2. Run the weights-free floors — live, on CPU

One cell. It loads the `no_op` and `random` policy specs and the `pusht` env spec from the repo's registries, then calls `run_cell_from_specs(...)` — the real eval orchestrator — for `N` episodes each on CPU. We pool the per-episode outcomes and attach a **Wilson 95% CI** with the public `embodimetry.stats.wilson_ci`.

`N_EPISODES = 20` keeps the whole cell around ~1–2 min on a free CPU. Bump it up for a tighter interval (the full sweep uses 250 episodes/cell).

In [ ]:
from pathlib import Path

from embodimetry.envs import EnvRegistry
from embodimetry.eval import run_cell_from_specs
from embodimetry.policies import PolicyRegistry
from embodimetry.stats import wilson_ci

CONFIGS = Path("embodimetry/configs")
policies = PolicyRegistry.from_yaml(CONFIGS / "policies.yaml")
envs = EnvRegistry.from_yaml(CONFIGS / "envs.yaml")

env_spec = envs.get("pusht")
N_EPISODES = 20  # ~1-2 min total on a free Colab CPU; raise for a tighter CI

print(f"{'policy':10}  {'N':>3}  {'success':>8}  {'rate':>6}  {'Wilson 95% CI':>18}")
print("-" * 54)
for name in ("no_op", "random"):
    spec = policies.get(name)
    result = run_cell_from_specs(
        spec,
        env_spec,
        seed_idx=0,
        n_episodes=N_EPISODES,
        device="cpu",  # no GPU needed for the weights-free floors
        record_video=False,  # skip MP4 encode for speed
    )
    n = len(result.episodes)
    successes = sum(1 for e in result.episodes if e.success)
    lo, hi = wilson_ci(successes, n, ci=0.95)
    ci = f"[{lo:.3f}, {hi:.3f}]"
    print(f"{name:10}  {n:>3}  {successes:>8}  {result.success_rate:>6.3f}  {ci:>18}")

You just produced **real numbers**, reproducibly: same seed → same per-episode outcomes. Both floors should sit near zero on PushT — which is exactly what you want from zero-skill baselines. If `no_op` or `random` scored high, the success criterion would be suspect. (PushT's coverage-window success rule is lax enough that `random` occasionally nudges the block into the goal; that is itself a documented finding — see `docs/SUCCESS_CRITERION_AUDIT.md`.)

The same `wilson_ci` you just called is what backs every cell on the leaderboard, so the floor you measured and the headline numbers are on one ruler.

## 3. Swap in a real pretrained policy

The *exact same call* runs a real checkpoint — the only difference is the policy spec carries a pinned `repo_id` + `revision_sha`, so `load_policy` downloads the weights from the Hub. PushT's pretrained policy is `lerobot/diffusion_pusht`:

```python
# Needs a GPU runtime + a checkpoint download (~minutes). Switch Colab to a GPU runtime first.
result = run_cell_from_specs(
    policies.get("diffusion_policy"),  # pinned repo_id + revision_sha in configs/policies.yaml
    envs.get("pusht"),
    seed_idx=0,
    n_episodes=10,
    device="cuda",                     # vision policies want a GPU
)
lo, hi = wilson_ci(
    sum(e.success for e in result.episodes), len(result.episodes), ci=0.95
)
print(result.success_rate, (lo, hi))
```

That is the entire surface: `(policy_spec, env_spec, seed) → CellResult`. Baselines and billion-parameter VLAs go through the same contract. The compatibility matrix (which policy runs on which env) lives in `configs/policies.yaml` under each policy's `env_compat`.

To run a single cell from the command line instead of a notebook:

```bash
python embodimetry/scripts/run_one.py --policy diffusion_policy --env pusht --seed 0 --n-episodes 10
```

## 4. See a rollout

Every published cell ships its rollout videos. Here is a real Diffusion-Policy success on PushT (looping GIF, rendered from the committed MP4):

In [ ]:
from IPython.display import Image, display

# Rendered into the repo we just cloned. The source MP4s live under
# embodimetry/docs/assets/rollouts/loops/ ; the looping GIFs under docs/assets/gifs/.
display(Image(filename="embodimetry/docs/assets/gifs/dp-pusht.gif"))

## Where to go next

- **Browse the leaderboard numbers (zero GPU):** `python embodimetry/examples/read_results.py` prints the v1 headline cells with Wilson CIs from a committed mini-parquet.
- **Read the methodology:** the [paper](https://github.com/thrmnn/embodimetry/blob/main/paper/main.tex) and [`docs/`](https://github.com/thrmnn/embodimetry/tree/main/docs).
- **Reproduce a published cell:** `make reproduce` (see [`docs/REPRODUCE.md`](https://github.com/thrmnn/embodimetry/blob/main/docs/REPRODUCE.md)).
- **Bring your own env:** [`docs/ENV_CONTRIBUTION_GUIDE.md`](https://github.com/thrmnn/embodimetry/blob/main/docs/ENV_CONTRIBUTION_GUIDE.md).

Star the repo if this was useful: <https://github.com/thrmnn/embodimetry>